# Hindustani Raga Classifier — pitch-contour approach (DeepSRGM-style)

**Why this exists**: `raga_classifier_training.ipynb` (raw log-mel spectrogram CNN) hit a real ceiling — two rounds of standard regularization (SpecAugment, weight decay, higher dropout) failed to move validation accuracy off ~2-5%, while training accuracy kept climbing. With only ~13-14 recordings per raga from a small pool of well-known performers, the likely cause is that a spectrogram CNN can latch onto *performer timbre/recording fingerprint* rather than the raga's actual melodic structure — an easier pattern to fit that happens to correlate with the label in training data but doesn't generalize to a different recording of the same raga by someone else.

**This notebook takes the input representation the academic literature actually uses for this problem** (DeepSRGM, Madhusudhan & Chowdhary 2018 — the same architecture family behind the Carnatic-only pretrained checkpoint rejected earlier for being the wrong tradition): a **tonic-normalized pitch contour**, not a spectrogram. Extracting only the melodic F0 trajectory relative to the piece's own tonic strips out timbre entirely — the model literally cannot see performer voice/instrument identity, only pitch movement over time, which forces it to learn melodic structure instead.

**Architecture** (mirrors DeepSRGM): quantized pitch-contour tokens → Embedding → GRU → attention pooling → FC → 61-way softmax. Separate from `raga_classifier_training.ipynb` on purpose — once both finish, compare real held-out test accuracy between them rather than assuming which is better.

**Same split, for a fair comparison**: identical file-level train/val/test split (same seed, same logic) as the spectrogram notebook, so a comparison between the two is apples-to-apples.

**Before running:**
1. Add Data → `suryamajumder/thaat-and-raga-forest-trf-dataset`.
2. GPU T4 (x2 fine). **Internet must be ON** for this one — unlike the other two notebooks, this one needs to `pip install essentia` (not part of Kaggle's base image, and not something we can vendor/preinstall). Confirmed this the hard way: the first run failed with `No module named 'essentia'` on every file because I'd verified essentia locally on my own machine and wrongly assumed it'd also be on Kaggle's image.
3. Resumable the same way: `EXISTING_CACHE_INPUT` / `EXISTING_CHECKPOINT_INPUT`, Save Version when done or interrupted.

In [ ]:
# --- Config ---
AUDIO_SAMPLE_RATE = 44100   # Essentia's pitch/tonic algorithms are tuned for this
MELODY_HOP_SIZE = 128       # PredominantPitchMelodia default -> ~344.5 frames/sec

# Cents-relative-to-tonic quantization, same formula as DeepSRGM
# (github.com/shubhlohiya/automatic-raga-recognition, src/dataset_preprocessing.ipynb):
# token = round(1200 * log2(pitch/tonic) * (QUANT_K/100)), clipped to [0, VOCAB_SIZE-1].
# Reusing their validated formula/vocab size rather than inventing a new
# quantization scheme blind. Token 0 doubles as "unvoiced/silence" AND the
# bottom of the pitch range (matches their own clip(0) behavior) — see the
# voiced-fraction gate below for why this doesn't collapse into confusion.
QUANT_K = 5
VOCAB_SIZE = 209

INPUT_LENGTH = 5000         # tokens per subsequence -> ~14.5s at this hop rate, matches DeepSRGM
MIN_VOICED_FRACTION = 0.5   # drop a candidate window if more than half its frames are unvoiced

# TRAIN vs EVAL windowing now DECOUPLED (2026-09-18) — real evidence forced
# this: the 2026-09-17 change below (overlapping windows) was applied
# uniformly to train AND val/test segments, and majority-vote test accuracy
# actually DROPPED (41.0% -> 32.8%) even though per-segment test accuracy
# went up slightly (21.3% -> 22.9%). Root cause: 50%-overlapping windows
# share up to half their tokens with their neighbors, so their prediction
# errors are correlated rather than independent — majority voting's entire
# benefit comes from averaging over INDEPENDENT errors, so correlated
# "votes" from overlapping windows undermine exactly the effect it depends
# on (and can make it worse than fewer-but-independent votes). Overlap is
# still a real win for TRAINING (more augmented examples from limited
# recordings, no independence requirement there) — it's specifically
# test-time majority voting that overlap hurts. So: keep overlap for
# training, go back to non-overlapping for anything used in majority voting
# (val AND test — val treated the same as test here, both are "evaluation,"
# not training).
TRAIN_SEGMENT_STRIDE = INPUT_LENGTH // 2   # 50% overlap — augmentation only, fine for training
EVAL_SEGMENT_STRIDE = INPUT_LENGTH         # NO overlap — independent votes for majority voting

# Overlapping windows + widened cap (2026-09-17, still applies to TRAIN_SEGMENT_STRIDE
# above). Owner's idea: take multiple clips from one recording (different
# content, same label) — already partially true (MAX_SEGMENTS_PER_FILE), but
# those windows were non-overlapping, capping at ~276s (4.6min) of the 480s
# already-analyzed window while the rest sat computed-but-unused. Checked
# the actual math before changing anything: naively doubling both stride
# and cap together (40 segs @ 50% overlap) just packs more REDUNDANT
# segments into that same ~276-283s span — not what was asked for. This
# config (50 segs @ 50% overlap) instead gets BOTH more segments (2.5x) AND
# wider real coverage (~356s, ~29% more of the recording actually used)
# than the previous 20-segment non-overlapping setup, at zero extra
# precompute cost (same already-computed 480s pitch contour, just sliced
# more thoroughly). MAX_SEGMENTS_PER_FILE is shared with EVAL_SEGMENT_STRIDE
# too, but never binds there in practice — non-overlapping 5000-token
# windows over a 480s analysis window naturally cap out around ~33
# segments/file, well under 50.
MAX_SEGMENTS_PER_FILE = 50
MAX_ANALYZE_SECONDS = 480

INTRO_SKIP_SECONDS = 20     # same rationale as the spectrogram notebook — trim likely
OUTRO_SKIP_SECONDS = 15     # spoken intro / tanpura tuning / applause before analysis

PRECOMPUTE_WORKERS = 4

EMBED_DIM = 128
# Architecture upgrade (2026-09-17): GRU made bidirectional (offline
# classification has the full sequence available, so there's no reason to
# withhold future context the way a streaming model would have to) and
# HIDDEN_SIZE dropped 256 -> 160 specifically to offset bidirectionality's
# ~2x parameter cost, so total capacity doesn't drift back toward the
# 768-hidden regime that overfit badly. Measured, not estimated: new
# architecture (bidir GRU + LayerNorm + attention/mean-pool concat, see
# RagaGRU below) comes to 418K params vs. the previous 364K — only +15%.
HIDDEN_SIZE = 160
RNN_DROPOUT = 0.3            # dropout on the GRU's output before attention pooling —
                              # kept, same reasoning as HIDDEN_SIZE above
BATCH_SIZE = 16              # smaller than the spectrogram model's 32 — 5000-step GRU
                              # backprop is far more memory-hungry per sample than a CNN
EPOCHS = 25
LR = 1e-3
EARLY_STOP_PATIENCE = 10
MAX_TRAIN_SECONDS = 8 * 3600

SEED = 42                    # SAME seed as raga_classifier_training.ipynb — keeps the
                              # file-level split identical across both notebooks
NUM_WORKERS = 2

SEGMENTS_CACHE_DIR = "/kaggle/working/pitch_cache"
CHECKPOINT_PATH = "/kaggle/working/pitch_last_checkpoint.pt"
BEST_MODEL_PATH = "/kaggle/working/pitch_model_best.pt"

# None again (2026-09-18): the cache key now depends on which stride was
# used (TRAIN_SEGMENT_STRIDE vs EVAL_SEGMENT_STRIDE per file), so a cache
# from before this change won't have any EVAL_SEGMENT_STRIDE-hashed entries
# for val/test files — a fresh (partial, cheap — only ~120 of 952 files)
# precompute is required regardless of what's attached here.
EXISTING_CACHE_INPUT = None

# Deliberately NOT pointing this at a prior checkpoint either: both
# HIDDEN_SIZE and the model's architecture itself (bidirectional GRU,
# LayerNorm, attention+mean-pool concat) changed this run, so no earlier
# checkpoint's weights are shape-compatible with this one regardless.
# Training starts fresh at epoch 1.
EXISTING_CHECKPOINT_INPUT = None

In [ ]:
import subprocess, time, os

def run(cmd, timeout, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired as e:
        print(e.stdout or "")
        raise RuntimeError(f"TIMED OUT after {time.time()-t0:.0f}s: {label}")
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}): {label}")
    print(f"--- OK ({time.time()-t0:.0f}s): {label}")
    return result

run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")
# Not part of Kaggle's base image (confirmed the hard way — the first run of
# this notebook failed with "No module named 'essentia'" on every single
# file, since I'd only verified it locally and wrongly assumed it'd carry
# over). Needs Internet ON.
run("pip install -q essentia", timeout=300, label="pip install essentia")

# Same numpy-ABI bug already hit and fixed in kaggle_batch_runner.ipynb, now
# defended against here too: essentia (like numba before) commonly pins an
# older numpy than Kaggle's preinstalled 2.x, and pip only swaps numpy's
# Python files, leaving the old numpy-2.x-ABI compiled binaries behind ->
# "dtype size changed" the first time anything touches numpy.random (e.g.
# importing pandas/sklearn). Fix: force-reinstall the EXACT version pip
# resolved (not latest) so its binaries are self-consistent, then verify
# every numpy-touching import used later in this notebook works cleanly —
# doing this BEFORE any other cell imports numpy/torch/sklearn for real
# means nothing downstream gets a chance to load a broken copy.
_pip_show = run("pip show numpy", timeout=30, label="read resolved numpy version")
_numpy_version = next(
    line.split(":", 1)[1].strip()
    for line in _pip_show.stdout.splitlines() if line.startswith("Version:")
)
print(f"Resolved numpy version: {_numpy_version}")
run(f"pip install -q --force-reinstall --no-deps numpy=={_numpy_version}", timeout=120,
    label="force-reinstall numpy binaries (pinned)")
run(
    'python -c "import numpy, torch, sklearn, essentia.standard as es; '
    'es.TonicIndianArtMusic(); es.PredominantPitchMelodia(); '
    'print(numpy.__version__, torch.__version__, \'essentia OK\')"',
    timeout=60,
    label="verify numpy/torch/sklearn/essentia all import and load cleanly",
)
os.makedirs(SEGMENTS_CACHE_DIR, exist_ok=True)

In [ ]:
# --- Bring in a prior session's cache/checkpoint, if resuming ---
import shutil

if EXISTING_CACHE_INPUT:
    if not os.path.isdir(EXISTING_CACHE_INPUT):
        raise RuntimeError(f"EXISTING_CACHE_INPUT={EXISTING_CACHE_INPUT!r} not found.")
    shutil.copytree(EXISTING_CACHE_INPUT, SEGMENTS_CACHE_DIR, dirs_exist_ok=True)
    print(f"Copied in {len(os.listdir(SEGMENTS_CACHE_DIR))} cached files from a prior session.")
else:
    print("No existing cache attached — starting precompute from scratch.")

if EXISTING_CHECKPOINT_INPUT:
    if not os.path.isfile(EXISTING_CHECKPOINT_INPUT):
        raise RuntimeError(f"EXISTING_CHECKPOINT_INPUT={EXISTING_CHECKPOINT_INPUT!r} not found.")
    shutil.copy(EXISTING_CHECKPOINT_INPUT, CHECKPOINT_PATH)
    print("Copied in a prior training checkpoint — will resume from it.")
else:
    print("No existing checkpoint attached — training will start from epoch 1.")

In [ ]:
# --- Locate the dataset and enumerate every recording — identical to the
# spectrogram notebook, so the two are comparable. ---
import glob
from collections import defaultdict

candidates = [c for c in glob.glob("/kaggle/input/**/Thaat and Raga Forest*", recursive=True) if os.path.isdir(c)]
if not candidates:
    raise RuntimeError("TRF dataset not found under /kaggle/input. Add Data > 'suryamajumder/thaat-and-raga-forest-trf-dataset'.")
DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

files_by_raga = defaultdict(list)
for path in glob.glob(os.path.join(DATASET_ROOT, "*", "*", "*.mp3")):
    parts = path.split(os.sep)
    files_by_raga[(parts[-3], parts[-2])].append(path)

ragas = sorted(files_by_raga.keys(), key=lambda k: k[1])
raga2idx = {raga_key: i for i, raga_key in enumerate(ragas)}
idx2raga = {i: {"thaat": t, "raga": r} for (t, r), i in raga2idx.items()}
print(f"{len(ragas)} ragas, {sum(len(v) for v in files_by_raga.values())} recordings total")

In [ ]:
import random
random.seed(SEED)

train_files, val_files, test_files = [], [], []
for raga_key, fs in files_by_raga.items():
    fs = sorted(fs)
    random.shuffle(fs)
    label = raga2idx[raga_key]
    if len(fs) >= 3:
        test_files.append((fs[0], label))
        val_files.append((fs[1], label))
        train_files.extend((f, label) for f in fs[2:])
    else:
        train_files.extend((f, label) for f in fs)

all_files = train_files + val_files + test_files
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

In [ ]:
# --- Precompute: tonic + pitch contour -> quantized token sequence, cached
# once per recording. Resumable and parallelized, same discipline as the
# spectrogram notebook (config baked into the cache key from the start —
# no legacy-migration needed since this cache is new).
#
# Stride now depends on the file's split (2026-09-18): train_files get
# TRAIN_SEGMENT_STRIDE (overlapping, for data augmentation), val_files +
# test_files get EVAL_SEGMENT_STRIDE (non-overlapping, for independent
# majority-vote votes) — see the config cell for why. _cache_path now takes
# the stride explicitly rather than reading a single global value, so the
# same file could in principle be cached under both strides (in practice
# each file only ever needs one, based on which split it's in). ---
import numpy as np
import librosa
import hashlib
from concurrent.futures import ProcessPoolExecutor, as_completed

def _cache_path(path, stride):
    config_key = (AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH,
                  stride, MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
                  INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS)
    h = hashlib.md5(f"{path}|{config_key}".encode()).hexdigest()
    return os.path.join(SEGMENTS_CACHE_DIR, f"{h}.npy")

def _precompute_one(args):
    (path, sr, hop_size, quant_k, vocab_size, input_length, stride, min_voiced_frac,
     max_segs, max_analyze_secs, intro_skip_secs, outro_skip_secs, out_path) = args
    if os.path.exists(out_path):
        return path, "cached", None
    try:
        import essentia.standard as es  # imported inside the worker — cheap, avoids pickling issues

        y, _ = librosa.load(path, sr=sr, mono=True)
        total_len = len(y)
        intro_skip = int(intro_skip_secs * sr)
        outro_skip = int(outro_skip_secs * sr)
        usable_start, usable_end = intro_skip, total_len - outro_skip
        if usable_end - usable_start < sr * 5:  # need at least a few seconds
            usable_start, usable_end = 0, total_len

        # Cap how much audio actually gets fed to the expensive pitch/tonic
        # algorithms — see MAX_ANALYZE_SECONDS comment in the config cell.
        max_len = int(max_analyze_secs * sr)
        if usable_end - usable_start > max_len:
            usable_end = usable_start + max_len

        y = y[usable_start:usable_end].astype(np.float32)

        tonic = es.TonicIndianArtMusic(sampleRate=sr)(y)
        pitch, _conf = es.PredominantPitchMelodia(sampleRate=sr, hopSize=hop_size)(y)

        voiced = pitch > 0
        tokens = np.zeros(len(pitch), dtype=np.int32)  # 0 = unvoiced/silence AND bottom of pitch range
        cents = 1200.0 * np.log2(pitch[voiced] / tonic)
        tokens[voiced] = np.clip(np.round(cents * (quant_k / 100.0)), 0, vocab_size - 1).astype(np.int32)

        segments = []
        for s in range(0, max(1, len(tokens) - input_length + 1), stride):
            chunk = tokens[s:s + input_length]
            if len(chunk) < input_length // 2:
                continue
            if voiced[s:s + len(chunk)].mean() < min_voiced_frac:
                continue  # mostly silence/unvoiced — likely a gap the intro/outro skip missed
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))
            if len(segments) >= max_segs:
                break

        if not segments:
            # Every window failed the voiced-fraction gate — fall back to the
            # single most-voiced window rather than dropping the recording.
            best_start, best_frac = 0, -1.0
            for s in range(0, max(1, len(tokens) - input_length + 1), stride):
                frac = voiced[s:s + input_length].mean()
                if frac > best_frac:
                    best_frac, best_start = frac, s
            chunk = tokens[best_start:best_start + input_length]
            if len(chunk) < input_length // 2:
                return path, "empty", None
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))

        np.save(out_path, np.stack(segments))
        return path, "done", None
    except Exception as e:
        return path, "error", str(e)

def _make_tasks(files, stride):
    return [(p, AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH, stride,
              MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
              INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS, _cache_path(p, stride)) for p, _ in files]

tasks = _make_tasks(train_files, TRAIN_SEGMENT_STRIDE) + _make_tasks(val_files + test_files, EVAL_SEGMENT_STRIDE)
already_cached = sum(1 for t in tasks if os.path.exists(t[-1]))
print(f"{already_cached}/{len(tasks)} already cached (resumed) — {len(tasks) - already_cached} left to process")

t0 = time.time()
done, errors = 0, []
with ProcessPoolExecutor(max_workers=PRECOMPUTE_WORKERS) as ex:
    futures = [ex.submit(_precompute_one, t) for t in tasks]
    for fut in as_completed(futures):
        path, status, err = fut.result()
        if status == "error":
            errors.append((path, err))
            print(f"  ERROR on {path}: {err}")
        done += 1
        if done % 20 == 0 or done == len(tasks):
            elapsed = time.time() - t0
            rate = elapsed / max(1, done - already_cached) if done > already_cached else None
            remaining = len(tasks) - done
            eta = f"{rate * remaining / 60:.1f}min" if rate else "n/a (still warming up)"
            print(f"  {done}/{len(tasks)} processed, {elapsed/60:.1f}min elapsed, ETA for rest: {eta}")

print(f"Precompute finished: {done} processed, {len(errors)} errors.")
if errors:
    print("Files with errors (will be skipped below):", [e[0] for e in errors])

In [ ]:
# --- Dataset: every cached token-sequence segment is its own example.
# `augment` (train only) randomly zeroes a few short spans — the sequence
# analogue of SpecAugment. Strengthened 2026-09-17 (2->3 masks, width
# 200->300) after the first real run showed clear overfitting (train_acc
# 74.5% vs val plateaued ~19-21%) — the original setting was deliberately
# mild since it was added pre-emptively with no evidence yet; now there is.
# `stride` param added 2026-09-18 (train vs eval windowing decoupled — see
# config cell): the dataset now needs to know which cache (overlapping
# TRAIN_SEGMENT_STRIDE or non-overlapping EVAL_SEGMENT_STRIDE) to read for
# a given file list, since both can now exist for the same file in principle. ---
import torch
from torch.utils.data import Dataset

def _load_cached(path, stride):
    cache_path = _cache_path(path, stride)
    if not os.path.exists(cache_path):
        return None
    return np.load(cache_path)

def _sequence_mask(tokens, n_masks=3, max_width=300):
    tokens = tokens.copy()
    n = len(tokens)
    for _ in range(n_masks):
        w = random.randint(0, max_width)
        if w == 0 or w >= n:
            continue
        start = random.randint(0, n - w)
        tokens[start:start + w] = 0
    return tokens

class FlattenedTokenDataset(Dataset):
    def __init__(self, file_label_pairs, stride, augment=False):
        self.augment = augment
        self.stride = stride
        self.index = []
        skipped = 0
        for path, label in file_label_pairs:
            cp = _cache_path(path, stride)
            if not os.path.exists(cp):
                skipped += 1
                continue
            n_segs = np.load(cp, mmap_mode="r").shape[0]
            self.index.extend((path, i, label) for i in range(n_segs))
        if skipped:
            print(f"  (skipping {skipped} files with no cache)")
        print(f"  {len(file_label_pairs) - skipped} files -> {len(self.index)} cached segments"
              f"{' (augmented)' if augment else ''}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        path, seg_idx, label = self.index[idx]
        segs = _load_cached(path, self.stride)
        tokens = segs[seg_idx].astype(np.int64)
        if self.augment:
            tokens = _sequence_mask(tokens)
        return torch.from_numpy(tokens), label

In [ ]:
import torch.nn as nn

class AttentionPool(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, time, hidden)
        weights = torch.softmax(self.score(x).squeeze(-1), dim=1)  # (batch, time)
        return torch.bmm(weights.unsqueeze(1), x).squeeze(1)       # (batch, hidden)

class RagaGRU(nn.Module):
    """
    Architecture upgrade (2026-09-17), each piece independently justified
    and validated locally (params measured, forward+backward tested) before
    landing here — see the HIDDEN_SIZE comment in the config cell:

    1. Bidirectional GRU (was unidirectional) — this is offline classification
       (the full recording is available before we classify it, not
       streaming), so there's no reason to withhold future context the way
       a real-time model would have to. Doubles the GRU's own parameter
       count, offset by dropping HIDDEN_SIZE 256->160.
    2. LayerNorm on the GRU output — cheap stabilization, negligible params.
    3. Attention pooling CONCATENATED with mean-pooling (not replaced).
       Mean-pooling adds ZERO trainable parameters and is a low-variance,
       hard-to-overfit signal, unlike the higher-capacity learned attention
       mechanism — a natural complement given how little data/class we have.

    Deliberately NOT done: a second GRU layer (adds capacity, contradicts
    the "less capacity helped" finding from the last two runs) or a
    Transformer-style replacement (self-attention architectures generally
    need MORE data than RNNs to train well — data scarcity is the whole
    problem here, not a reason to make it worse).
    """
    def __init__(self, num_classes, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_size=HIDDEN_SIZE, rnn_dropout=RNN_DROPOUT):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(hidden_size * 2)
        self.rnn_dropout = nn.Dropout(rnn_dropout)
        self.attention = AttentionPool(hidden_size * 2)
        pooled_size = hidden_size * 2 * 2  # attention output + mean-pool output, concatenated
        fc1_size = max(64, pooled_size // 4)
        self.fc1 = nn.Linear(pooled_size, fc1_size)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(fc1_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(self.embeddings(x))
        out = self.norm(out)
        out = self.rnn_dropout(out)
        attn_out = self.attention(out)
        mean_out = out.mean(dim=1)
        pooled = torch.cat([attn_out, mean_out], dim=1)
        out = self.dropout(self.relu(self.fc1(pooled)))
        return self.fc2(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type != "cuda":
    raise RuntimeError("No GPU detected. Check Notebook Settings > Accelerator.")

_param_count = sum(p.numel() for p in RagaGRU(num_classes=len(ragas)).parameters())
print(f"model params: {_param_count:,} (was 364,222 before this architecture change)")

# Every input is padded/truncated to the exact same INPUT_LENGTH, so cuDNN
# never has to re-tune for a new shape — free speedup with zero downside here.
torch.backends.cudnn.benchmark = True

In [ ]:
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

# stride explicit per split now (2026-09-18) — train gets the overlapping
# augmentation cache, val gets the non-overlapping eval cache (same
# treatment as test_files below, since val here plays the same
# "independent votes" role for early-stopping decisions).
train_dataset = FlattenedTokenDataset(train_files, stride=TRAIN_SEGMENT_STRIDE, augment=True)
val_dataset = FlattenedTokenDataset(val_files, stride=EVAL_SEGMENT_STRIDE, augment=False)

train_seg_labels = np.array([label for _, _, label in train_dataset.index])
class_weights_arr = compute_class_weight("balanced", classes=np.arange(len(ragas)), y=train_seg_labels)
class_weights = torch.tensor(class_weights_arr, dtype=torch.float32).to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

model = RagaGRU(num_classes=len(ragas)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

# Mixed precision (2026-09-17) — real speed/memory win on the T4's tensor
# cores. Control flow (autocast -> scale -> unscale_ -> clip -> step ->
# update) validated locally against a CPU stand-in with the scaler disabled
# before adding here, specifically because unscale_ must run BEFORE
# clip_grad_norm_ or you silently clip the scaled (artificially large)
# gradients instead of the real ones.
scaler = torch.amp.GradScaler("cuda")

start_epoch = 1
best_val_acc = 0.0
epochs_no_improve = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    try:
        model.load_state_dict(ckpt["model"])
    except RuntimeError as e:
        # Most likely cause: a checkpoint left over in /kaggle/working from
        # before today's architecture change (bidirectional GRU + LayerNorm +
        # attention/mean-pool concat), rather than a fresh session. That old
        # checkpoint's tensor shapes don't match this model at all, so fail
        # loudly with the fix instead of surfacing a raw shape-mismatch trace.
        raise RuntimeError(
            f"Checkpoint at {CHECKPOINT_PATH} doesn't match the current model "
            f"architecture — almost certainly a leftover from before this "
            f"session's architecture change. Delete {CHECKPOINT_PATH} and "
            f"{BEST_MODEL_PATH} (or start a fresh Kaggle session) and re-run. "
            f"Original error: {e}"
        )
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    epochs_no_improve = ckpt["epochs_no_improve"]
    print(f"Resumed from checkpoint: starting at epoch {start_epoch}, best_val_acc so far={best_val_acc:.3f}")
else:
    print("No checkpoint found — starting fresh at epoch 1.")

print(f"train batches/epoch: {len(train_loader)}  val batches: {len(val_loader)}")

In [ ]:
# --- Training loop. Gradient clipping added 2026-09-17 (was missing) — GRUs
# over long sequences (5000 timesteps here) are well-known to be prone to
# exploding gradients, which can silently spike loss to NaN and waste the
# whole run with no warning. Standard, essentially required for RNN
# training at this sequence length, not just a nice-to-have. Mixed
# precision (autocast + GradScaler) added the same day for the T4 speed win
# — note unscale_() runs before clip_grad_norm_, not after: clipping the
# still-scaled gradients would clip against the wrong magnitude entirely. ---
t_start = time.time()
GRAD_CLIP_NORM = 5.0

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * x.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += x.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast("cuda"):
                out = model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += x.size(0)

    train_acc = train_correct / max(1, train_total)
    val_acc = val_correct / max(1, val_total)
    scheduler.step(val_acc)
    elapsed = time.time() - t_start
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss/train_total:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}  elapsed={elapsed/60:.1f}min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> new best (val_acc={val_acc:.3f}), best-model checkpoint saved")
    else:
        epochs_no_improve += 1

    torch.save({
        "epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
        "best_val_acc": best_val_acc, "epochs_no_improve": epochs_no_improve,
    }, CHECKPOINT_PATH)

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"No val improvement for {EARLY_STOP_PATIENCE} epochs — stopping early.")
        break
    if elapsed > MAX_TRAIN_SECONDS:
        print(f"Hit MAX_TRAIN_SECONDS ({MAX_TRAIN_SECONDS}s) safety limit — stopping. "
              f"Save Version now and resume via EXISTING_CHECKPOINT_INPUT next session.")
        break

print(f"Best val accuracy: {best_val_acc:.3f}")

In [ ]:
# --- Held-out TEST evaluation: per-segment AND per-file majority-vote accuracy.
# Uses EVAL_SEGMENT_STRIDE (non-overlapping) segments (2026-09-18) — see the
# config cell: majority voting needs independent votes, and this is exactly
# what regressed majority-vote accuracy (41.0% -> 32.8%) when test segments
# started overlapping too.
# Also saves full per-segment softmax probabilities (2026-09-17) — not needed
# for the accuracy numbers themselves, but required later to build an
# ensemble with raga_classifier_hmm.ipynb's per-segment log-likelihoods,
# once both notebooks have real standalone results to combine. ---
if os.path.exists(BEST_MODEL_PATH):
    try:
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    except RuntimeError as e:
        raise RuntimeError(
            f"{BEST_MODEL_PATH} doesn't match the current model architecture — "
            f"same stale-checkpoint issue as the training cell above. Delete "
            f"{CHECKPOINT_PATH} and {BEST_MODEL_PATH} and re-run from a fresh "
            f"session. Original error: {e}"
        )
model.eval()

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
seg_correct, seg_total = 0, 0
file_correct, file_total = 0, 0
all_test_probs = []  # list of (file_path, true_label, probs: np.ndarray shape (n_segments, num_classes))

with torch.no_grad():
    for path, label in test_files:
        segs = _load_cached(path, EVAL_SEGMENT_STRIDE)
        if segs is None:
            continue
        x = torch.from_numpy(segs.astype(np.int64)).to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(1)
        all_test_probs.append((path, label, probs))
        seg_correct += int((preds == label).sum())
        seg_total += len(preds)
        majority = int(np.bincount(preds).argmax())
        file_total += 1
        per_class_total[label] += 1
        if majority == label:
            file_correct += 1
            per_class_correct[label] += 1

print(f"Per-segment test accuracy: {seg_correct/max(1,seg_total):.3f}  ({seg_correct}/{seg_total})")
print(f"Per-file majority-vote test accuracy: {file_correct/max(1,file_total):.3f}  ({file_correct}/{file_total})")
print()
print("Per-class (majority-vote) breakdown:")
for label in sorted(per_class_total):
    print(f"  {idx2raga[label]['raga']:35s} {per_class_correct[label]}/{per_class_total[label]}")

In [ ]:
import json, pickle

with open("/kaggle/working/pitch_raga_classes.json", "w") as f:
    json.dump(idx2raga, f, indent=2, ensure_ascii=False)
with open("/kaggle/working/pitch_preprocessing_config.json", "w") as f:
    json.dump({"audio_sample_rate": AUDIO_SAMPLE_RATE, "melody_hop_size": MELODY_HOP_SIZE,
               "quant_k": QUANT_K, "vocab_size": VOCAB_SIZE, "input_length": INPUT_LENGTH}, f, indent=2)
with open("/kaggle/working/pitch_results.json", "w") as f:
    json.dump({
        "best_val_acc": best_val_acc,
        "test_segment_acc": seg_correct / max(1, seg_total),
        "test_majority_vote_acc": file_correct / max(1, file_total),
        "num_ragas": len(ragas),
        "epochs_completed": epoch,
    }, f, indent=2)
with open("/kaggle/working/pitch_test_probs.pkl", "wb") as f:
    pickle.dump(all_test_probs, f)

print("Saved: pitch_model_best.pt, pitch_last_checkpoint.pt, pitch_cache/, pitch_raga_classes.json, "
      "pitch_preprocessing_config.json, pitch_results.json, pitch_test_probs.pkl (for a later ensemble)")
print("Click 'Save Version' now — whether or not training finished — to persist the cache and checkpoint for a resumed run.")